# 🌳 NYC Street Tree Census — Full Data Analysis Pipeline
> **Dataset:** 2015 Street Tree Census — Tree Data  
> **Stages:** Import → Exploration → Cleaning → Visualization  
> **Author:** Data Analysis Notebook  
> **Purpose:** End-to-end professional workflow from raw data to insights

---
## 📋 Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Data](#2-load-data)
3. [Data Exploration](#3-data-exploration)
   - 3.1 Shape, Columns & dtypes
   - 3.2 Head / Tail Preview
   - 3.3 Statistical Summary
   - 3.4 Missing Values Analysis
   - 3.5 Unique Value Counts
   - 3.6 Display Options
4. [Select Relevant Columns](#4-select-relevant-columns)
5. [Data Cleaning](#5-data-cleaning)
   - 5.1 Handle Missing Values
   - 5.2 Fix Wrong / Outlier Data
   - 5.3 Remove Duplicates
   - 5.4 Reset Index
   - 5.5 Data Type Conversion
6. [Post-Cleaning Validation](#6-post-cleaning-validation)
7. [Data Visualization](#7-data-visualization)
8. [Export Clean Data](#8-export-clean-data)
---

## 1. Setup & Imports

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Global plot style ────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries loaded successfully.')

---
## 2. Load Data

In [ ]:
# ── Load the raw CSV ─────────────────────────────────────────────────────────
FILE_PATH = '2015_Street_Tree_Census_-_Tree_Data_20241224.csv'

df_raw = pd.read_csv(FILE_PATH)

print(f'✅ Data loaded successfully!')
print(f'   Rows    : {df_raw.shape[0]:,}')
print(f'   Columns : {df_raw.shape[1]:,}')

---
## 3. Data Exploration

### 3.1 Shape, Columns & Data Types

In [ ]:
# ── Overall shape ────────────────────────────────────────────────────────────
print('── Shape ──────────────────────────────')
print(f'  Rows    : {df_raw.shape[0]:,}')
print(f'  Columns : {df_raw.shape[1]:,}')

print('\n── All Column Names ────────────────────')
print(df_raw.columns.tolist())

print('\n── Data Types ──────────────────────────')
print(df_raw.dtypes)

In [ ]:
# ── Concise info table: dtypes + non-null counts ─────────────────────────────
df_raw.info()

### 3.2 Head / Tail Preview

In [ ]:
# ── Show display options so all columns are visible ───────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print('── First 5 Rows ────────────────────────')
df_raw.head()

In [ ]:
print('── Last 5 Rows ─────────────────────────')
df_raw.tail()

### 3.3 Statistical Summary

In [ ]:
# ── Numeric columns ──────────────────────────────────────────────────────────
print('── Numeric Statistics ──────────────────')
df_raw.describe()

In [ ]:
# ── Categorical columns ──────────────────────────────────────────────────────
print('── Categorical Statistics ──────────────')
df_raw.describe(include='object')

### 3.4 Missing Values Analysis

In [ ]:
# ── Count & percentage of nulls per column ───────────────────────────────────
missing_count = df_raw.isna().sum()
missing_pct   = (missing_count / len(df_raw) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %'    : missing_pct
}).sort_values('Missing Count', ascending=False)

print('── Missing Values Summary ──────────────')
missing_summary[missing_summary['Missing Count'] > 0]

### 3.5 Unique Value Counts per Column

In [ ]:
# ── How many distinct values does each column have? ──────────────────────────
print('── Unique Values per Column ────────────')
print(df_raw.nunique().sort_values())

In [ ]:
# ── Value counts for key categorical columns ─────────────────────────────────
for col in ['status', 'health', 'curb_loc', 'borough']:
    if col in df_raw.columns:
        print(f'\n── {col} ──')
        print(df_raw[col].value_counts(dropna=False))

### 3.6 Display Options Helper

In [ ]:
# ── Restore limited display for large DataFrames ─────────────────────────────
pd.set_option('display.max_rows',    100)
pd.set_option('display.max_columns', 20)

# To restore unlimited display run:
#   pd.set_option('display.max_columns', None)
#   pd.set_option('display.max_rows',    None)

print('Display options set: 100 rows × 20 columns')

---
## 4. Select Relevant Columns

In [ ]:
# ── Keep only the analytically relevant columns ───────────────────────────────
COLS_TO_KEEP = [
    'tree_id', 'tree_dbh', 'stump_diam',
    'curb_loc', 'status', 'health',
    'spc_latin', 'steward', 'sidewalk', 'problems',
    'root_stone', 'root_grate', 'root_other',
    'trunk_wire', 'trnk_light', 'trnk_other',
    'brch_light', 'brch_shoe', 'brch_other'
]

df = df_raw[COLS_TO_KEEP].copy()

print(f'✅ Kept {len(COLS_TO_KEEP)} columns out of {df_raw.shape[1]}')
print(f'   Shape : {df.shape}')
df.head(3)

---
## 5. Data Cleaning

### 5.1 Handle Missing Values

| Column | Missing | Strategy | Reason |
|--------|---------|----------|--------|
| `health` | ~31k | Fill with `'Unknown'` | NaN = Dead/Stump rows — label them |
| `spc_latin` | ~31k | Fill with `'Unknown'` | Dead/Stump have no species |
| `sidewalk` | ~31k | Fill with `'Unknown'` | Same rows |
| `steward` | ~519k | Fill with `'None'` | No steward recorded |
| `problems` | ~458k | Fill with `'None'` | No problem recorded |

In [ ]:
# ── Before ───────────────────────────────────────────────────────────────────
print('── Missing Values BEFORE Cleaning ──────')
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# ── Strategy A: fill categorical NaNs with meaningful labels ─────────────────
df.fillna({
    'health'   : 'Unknown',   # Dead/Stump trees have no health rating
    'spc_latin': 'Unknown',   # Dead/Stump trees have no species
    'sidewalk' : 'Unknown',   # Not applicable for non-alive trees
    'steward'  : 'None',      # No steward assigned
    'problems' : 'None'       # No problem reported
}, inplace=True)

# ── Strategy B: numeric — fill with column median (robust to outliers) ────────
#   (tree_dbh and stump_diam have 0 nulls here, but shown as best practice)
for col in ['tree_dbh', 'stump_diam']:
    if df[col].isna().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print('✅ Missing values filled.')

In [ ]:
# ── Drop any rows that are STILL completely empty (safety net) ────────────────
before = len(df)
df.dropna(how='all', inplace=True)
print(f'Rows dropped (all-NaN): {before - len(df)}')

### 5.2 Fix Wrong / Outlier Data

In [ ]:
# ── Inspect tree diameter distribution ──────────────────────────────────────
print('── tree_dbh stats ──────────────────────')
print(df['tree_dbh'].describe())
print(f'\nRows with tree_dbh > 450 (likely erroneous): '
      f"{(df['tree_dbh'] > 450).sum()}")

In [ ]:
# ── Cap extreme outliers at the 99th percentile ───────────────────────────────
#   (rather than deleting — preserves row count)
cap_dbh = df['tree_dbh'].quantile(0.99)
df.loc[df['tree_dbh'] > cap_dbh, 'tree_dbh'] = cap_dbh

print(f'✅ tree_dbh capped at {cap_dbh} (99th percentile).')

In [ ]:
# ── Fix a single specific known-bad value (example pattern) ──────────────────
#   df.loc[ROW_INDEX, 'COLUMN'] = CORRECT_VALUE
#
# Example (uncomment and customise):
# df.loc[7, 'tree_dbh'] = 45

# ── Fix via loop (clip any remaining outliers) ────────────────────────────────
for x in df.index:
    if df.loc[x, 'tree_dbh'] > 450:
        df.loc[x, 'tree_dbh'] = 450

print('✅ Loop-based outlier fix applied.')

### 5.3 Remove Duplicates

In [ ]:
# ── How many duplicate rows are there? ───────────────────────────────────────
n_dups = df.duplicated().sum()
print(f'Duplicate rows found : {n_dups}')

# ── Show duplicated rows (keep=False marks ALL copies) ────────────────────────
if n_dups > 0:
    print(df[df.duplicated(keep=False)].head(10))

In [ ]:
# ── Remove duplicates: keep='first' retains the first occurrence ──────────────
before = len(df)
df.drop_duplicates(keep='first', inplace=True)
print(f'✅ Rows removed: {before - len(df)}')
print(f'   Rows remaining: {len(df):,}')

# ── To deduplicate on specific key columns only: ──────────────────────────────
# df.drop_duplicates(subset=['tree_id'], keep='first', inplace=True)

### 5.4 Reset Index

In [ ]:
# ── After drops/deduplication the index has gaps — reset it ──────────────────
df.reset_index(drop=True, inplace=True)
print(f'✅ Index reset. Index range: 0 → {len(df)-1:,}')

### 5.5 Data Type Conversion

In [ ]:
# ── Ensure numeric columns are correct dtype ──────────────────────────────────
df['tree_id']    = df['tree_id'].astype(int)
df['tree_dbh']   = df['tree_dbh'].astype(float)
df['stump_diam'] = df['stump_diam'].astype(float)

# ── Convert Yes/No binary columns to boolean ──────────────────────────────────
binary_cols = [
    'root_stone', 'root_grate', 'root_other',
    'trunk_wire', 'trnk_light', 'trnk_other',
    'brch_light', 'brch_shoe',  'brch_other'
]
for col in binary_cols:
    df[col] = df[col].map({'Yes': True, 'No': False})

print('✅ Data types converted.')
print(df.dtypes)

---
## 6. Post-Cleaning Validation

In [ ]:
print('══════════════════════════════════════')
print('  POST-CLEANING DATA QUALITY REPORT  ')
print('══════════════════════════════════════')
print(f'  Total rows        : {len(df):,}')
print(f'  Total columns     : {df.shape[1]}')
print(f'  Remaining NaNs    : {df.isna().sum().sum()}')
print(f'  Duplicate rows    : {df.duplicated().sum()}')
print(f'  tree_dbh max      : {df["tree_dbh"].max()}')
print(f'  tree_dbh min      : {df["tree_dbh"].min()}')
print('══════════════════════════════════════')

# ── Value counts for all binary (bool) problem columns ────────────────────────
print('\n── Problem Columns: Yes/No Counts ──────')
df[binary_cols].apply(pd.Series.value_counts)

---
## 7. Data Visualization

### 7.1 Tree Health Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

health_counts = df['health'].value_counts()

# Bar chart
health_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', len(health_counts)))
axes[0].set_title('Tree Health — Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Health Status')
axes[0].set_ylabel('Number of Trees')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(health_counts, labels=health_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('muted', len(health_counts)), startangle=140)
axes[1].set_title('Tree Health — Share', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 7.2 Tree Status Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
status_counts = df['status'].value_counts()

status_counts.plot(kind='barh', ax=ax, color=sns.color_palette('Set2', len(status_counts)))
ax.set_title('Tree Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Trees')
ax.set_ylabel('Status')
for i, v in enumerate(status_counts):
    ax.text(v + 1000, i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 7.3 Top 15 Tree Species

In [ ]:
top_species = (
    df[df['spc_latin'] != 'Unknown']['spc_latin']
    .value_counts()
    .head(15)
)

fig, ax = plt.subplots(figsize=(14, 6))
top_species.plot(kind='bar', ax=ax, color=sns.color_palette('tab20', 15))
ax.set_title('Top 15 Tree Species (by count)', fontsize=14, fontweight='bold')
ax.set_xlabel('Species (Latin Name)')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

### 7.4 Tree Diameter (DBH) Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
alive_dbh = df[df['status'] == 'Alive']['tree_dbh']
axes[0].hist(alive_dbh, bins=50, color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_title('DBH Distribution (Alive Trees)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tree Diameter at Breast Height (inches)')
axes[0].set_ylabel('Count')

# Box plot by health
health_order = ['Good', 'Fair', 'Poor', 'Unknown']
plot_data = df[df['health'].isin(health_order)]
plot_data.boxplot(column='tree_dbh', by='health', ax=axes[1],
                  positions=range(len(health_order)),
                  patch_artist=True)
axes[1].set_title('DBH by Health Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Health')
axes[1].set_ylabel('DBH')
plt.suptitle('')  # Remove auto-title from boxplot

plt.tight_layout()
plt.show()

### 7.5 Problem Columns — Yes/No Heatmap

In [ ]:
# ── Build count matrix ────────────────────────────────────────────────────────
problem_counts = df[binary_cols].apply(pd.Series.value_counts).fillna(0).astype(int)
print(problem_counts)

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(
    problem_counts,
    annot=True, fmt=',d', cmap='YlOrRd',
    linewidths=0.5, ax=ax
)
ax.set_title('Problem Indicators — Count Heatmap (True = Issue Present)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.6 Steward Activity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
steward_counts = df['steward'].value_counts()

steward_counts.plot(kind='bar', ax=ax, color=sns.color_palette('coolwarm', len(steward_counts)))
ax.set_title('Stewardship Level Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Steward Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

---
## 8. Export Clean Data

In [ ]:
# ── Save cleaned DataFrame ───────────────────────────────────────────────────
OUTPUT_PATH = 'nyc_trees_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Clean data exported → {OUTPUT_PATH}')
print(f'   Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# ── Final preview of clean data ──────────────────────────────────────────────
print('── Final Clean DataFrame ───────────────')
df.head(10)